##  Reading Files and Feature Creation

In [1]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path.cwd()
ARTIFACTS = PROJECT_DIR / "artifacts"
MODELS = PROJECT_DIR / "models"
MODELS.mkdir(exist_ok=True)

train = pd.read_csv(ARTIFACTS / "03_train.csv")
validation = pd.read_csv(ARTIFACTS / "03_validation.csv")
test = pd.read_csv(ARTIFACTS / "03_test.csv")

def add_features(df):
    df = df.copy()
    date = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
    df["purchase_year"] = date.dt.year
    df["purchase_month"] = date.dt.month
    df["purchase_dayofweek"] = date.dt.dayofweek
    df["purchase_hour"] = date.dt.hour
    return df

train = add_features(train)
validation = add_features(validation)
test = add_features(test)

## Feature Selection and Preventing Data Leakage










In [2]:
target = "is_late"

candidate_features = [
    "customer_state",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "item_count",
    "total_price",
    "total_freight",
    "unique_products",
    "unique_sellers",
    "total_payment",
    "payment_count",
    "max_installments"
]

feature_cols = [col for col in candidate_features if col in train.columns]

X_train = train[feature_cols]
y_train = train[target]
X_validation = validation[feature_cols]
y_validation = validation[target]
X_test = test[feature_cols]
y_test = test[target]

print("Features used:")
print(feature_cols)

Features used:
['customer_state', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'item_count', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'total_payment', 'payment_count', 'max_installments']


## Feature Preparation
     

In [4]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# fit على training فقط، ثم transform للباقي
X_train_ready = preprocessor.fit_transform(X_train)
X_validation_ready = preprocessor.transform(X_validation)
X_test_ready = preprocessor.transform(X_test)

print("Train matrix:", X_train_ready.shape)
print("Validation matrix:", X_validation_ready.shape)
print("Test matrix:", X_test_ready.shape)

C:\Users\HP\AppData\Local\Temp\ipykernel_18912\4289928263.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


Train matrix: (57885, 39)
Validation matrix: (19295, 39)
Test matrix: (19296, 39)


## Saving Artifacts Without Parquet


In [5]:
joblib.dump(preprocessor, MODELS / "preprocessor.joblib")
joblib.dump(feature_cols, MODELS / "feature_list.joblib")

# joblib يحفظ المصفوفات، سواء كانت sparse أو dense
joblib.dump(X_train_ready, ARTIFACTS / "05_X_train.joblib")
joblib.dump(X_validation_ready, ARTIFACTS / "05_X_validation.joblib")
joblib.dump(X_test_ready, ARTIFACTS / "05_X_test.joblib")

# حفظ الـlabels كـCSV
pd.Series(y_train).to_csv(ARTIFACTS / "05_y_train.csv", index=False)
pd.Series(y_validation).to_csv(ARTIFACTS / "05_y_validation.csv", index=False)
pd.Series(y_test).to_csv(ARTIFACTS / "05_y_test.csv", index=False)

print("Feature artifacts saved")

Feature artifacts saved
